# Statistiques descriptives — Base GDELT GKG (2015 → aujourd'hui)

Ce notebook explore la base Parquet hebdomadaire produite par `gdelt_weekly_pipeline.py`.

**Rappel du schéma de chaque fichier `gdelt_YYYY-Www.parquet` :**

| Colonne | Description |
|---|---|
| `GKGRECORDID` | identifiant unique de l'enregistrement GKG |
| `DATE` | horodatage GDELT au format `YYYYMMDDHHMMSS` (string) |
| `SourceCollectionIdentifier` | canal de collecte GDELT (1=WEB, 2=CITATIONONLY, 3=CORE, 4=DTIC, 5=JSTOR, 6=NONTEXTUALSOURCE — d'après la documentation GDELT) |
| `DocumentIdentifier` | URL de l'article source |
| `EnhancedThemes` | thèmes GKG (format `THEME,offset;THEME,offset;...`) |
| `EnhancedLocations` | lieux géolocalisés détectés |
| `Persons` | personnes citées (séparées par `;`) |
| `Organizations` | organisations citées (séparées par `;`) |
| `TranslationInfo` | infos de traduction (si article non anglophone) |
| `Tone` | tonalité moyenne de l'article (échelle GDELT, ~-10 à +10) |
| `WordCount` | nombre de mots approximatif de l'article |
| `translated` | 1 si l'article provient de la master list translingue, 0 sinon |
| `SourceCommonName_ID` | identifiant entier du média, à mapper via `gdelt_sources_mapping.json` |

On utilise **DuckDB** pour interroger directement les fichiers Parquet sans tout charger en RAM (la base peut représenter plusieurs dizaines de millions de lignes sur plusieurs années), et **pandas/matplotlib/seaborn** pour les agrégations plus fines et les visualisations.


## 1. Installation & imports

In [ ]:
# Décommenter si besoin :
# !pip install duckdb pandas pyarrow matplotlib seaborn --quiet

import os
import glob
import json
from pathlib import Path

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 2. Configuration & inventaire des fichiers

In [ ]:
# Adapter ces chemins si nécessaire
DATA_DIR = Path("./gdelt_parquet_db")
SOURCE_MAP_PATH = Path("./gdelt_sources_mapping.json")

parquet_files = sorted(glob.glob(str(DATA_DIR / "gdelt_*.parquet")))
print(f"{len(parquet_files)} fichiers parquet (semaines) trouvés.")

if parquet_files:
    print("Premier fichier :", os.path.basename(parquet_files[0]))
    print("Dernier fichier :", os.path.basename(parquet_files[-1]))
    total_size_gb = sum(os.path.getsize(f) for f in parquet_files) / (1024**3)
    print(f"Taille totale sur disque : {total_size_gb:.2f} Go")
else:
    print("⚠️ Aucun fichier trouvé — vérifie DATA_DIR.")


## 3. Connexion DuckDB et vue unifiée sur l'ensemble des semaines

In [ ]:
con = duckdb.connect()

glob_pattern = str(DATA_DIR / "gdelt_*.parquet")
con.execute(f"""
    CREATE OR REPLACE VIEW gkg AS
    SELECT * FROM read_parquet('{glob_pattern}')
""")

n_rows = con.execute("SELECT COUNT(*) FROM gkg").fetchone()[0]
print(f"Nombre total d'articles (lignes GKG) : {n_rows:,}")


In [ ]:
# Schéma effectif de la base
con.execute("DESCRIBE gkg").df()


In [ ]:
date_range = con.execute("SELECT MIN(DATE) AS min_date, MAX(DATE) AS max_date FROM gkg").fetchone()
print(f"Période couverte : {date_range[0]} → {date_range[1]}")


## 4. Describe des variables numériques (`Tone`, `WordCount`, `translated`)

In [ ]:
summary = con.execute("""
    SUMMARIZE SELECT Tone, WordCount, translated FROM gkg
""").df()
summary


In [ ]:
trans_ratio = con.execute("""
    SELECT translated,
           COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM gkg
    GROUP BY translated
""").df()
print("Répartition articles natifs (0) vs traduits (1) :")
trans_ratio


In [ ]:
collection_dist = con.execute("""
    SELECT SourceCollectionIdentifier, COUNT(*) AS n
    FROM gkg
    GROUP BY SourceCollectionIdentifier
    ORDER BY n DESC
""").df()
collection_dist


## 5. Journaux (sources) les plus représentés

In [ ]:
with open(SOURCE_MAP_PATH, "r", encoding="utf-8") as f:
    source_map = json.load(f)

id_to_source = source_map["id_to_source"]
src_df = pd.DataFrame({
    "SourceCommonName_ID": [int(k) for k in id_to_source.keys()],
    "SourceCommonName": list(id_to_source.values()),
})
print(f"{len(src_df):,} sources distinctes référencées dans le dictionnaire.")


In [ ]:
con.register("src_map", src_df)

top_sources = con.execute("""
    SELECT m.SourceCommonName, COUNT(*) AS n_articles
    FROM gkg g
    JOIN src_map m ON g.SourceCommonName_ID = m.SourceCommonName_ID
    GROUP BY m.SourceCommonName
    ORDER BY n_articles DESC
    LIMIT 30
""").df()
top_sources


In [ ]:
plt.figure(figsize=(10, 8))
sns.barplot(data=top_sources, y="SourceCommonName", x="n_articles", color="steelblue")
plt.title("Top 30 des sources (médias) les plus représentées")
plt.xlabel("Nombre d'articles")
plt.ylabel("")
plt.tight_layout()
plt.show()


In [ ]:
n_distinct_sources_used = con.execute("""
    SELECT COUNT(DISTINCT SourceCommonName_ID) FROM gkg
""").fetchone()[0]
print(f"Nombre de médias distincts apparaissant réellement dans la base : {n_distinct_sources_used:,}")


## 6. Volume d'articles par jour

On reconstruit le jour calendaire à partir des 8 premiers caractères de `DATE` (`YYYYMMDD`).


In [ ]:
articles_per_day = con.execute("""
    SELECT STRPTIME(SUBSTR(DATE, 1, 8), '%Y%m%d')::DATE AS day,
           COUNT(*) AS n_articles
    FROM gkg
    GROUP BY day
    ORDER BY day
""").df()

articles_per_day["day"] = pd.to_datetime(articles_per_day["day"])
articles_per_day = articles_per_day.set_index("day")

print(f"Nombre de jours couverts par la collecte : {len(articles_per_day):,}")
print(f"Total d'articles                          : {articles_per_day['n_articles'].sum():,.0f}")
print(f"Moyenne d'articles / jour                 : {articles_per_day['n_articles'].mean():,.1f}")
print(f"Médiane d'articles / jour                 : {articles_per_day['n_articles'].median():,.1f}")
print(f"Écart-type / jour                         : {articles_per_day['n_articles'].std():,.1f}")
print(f"Min / jour                                : {articles_per_day['n_articles'].min():,.0f}")
print(f"Max / jour                                : {articles_per_day['n_articles'].max():,.0f}")


In [ ]:
# Détection de jours manquants dans la collecte (utile pour évaluer la qualité du pipeline)
full_range = pd.date_range(articles_per_day.index.min(), articles_per_day.index.max(), freq="D")
missing_days = full_range.difference(articles_per_day.index)

print(f"Jours sans aucune donnée sur la période : {len(missing_days)} / {len(full_range)}")
if len(missing_days) > 0:
    print("Premiers jours manquants :", list(missing_days[:15].strftime('%Y-%m-%d')))


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
articles_per_day["n_articles"].plot(ax=ax, linewidth=0.7, color="steelblue")
articles_per_day["n_articles"].rolling(30, min_periods=1).mean().plot(
    ax=ax, linewidth=2, color="darkorange", label="Moyenne mobile 30 jours"
)
ax.set_title("Nombre d'articles GKG collectés par jour")
ax.set_ylabel("Articles / jour")
ax.set_xlabel("Date")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
monthly_volume = articles_per_day["n_articles"].resample("MS").sum()

fig, ax = plt.subplots(figsize=(14, 5))
monthly_volume.plot(ax=ax, marker="o", markersize=3, color="seagreen")
ax.set_title("Volume mensuel total d'articles GKG")
ax.set_ylabel("Articles / mois")
plt.tight_layout()
plt.show()


In [ ]:
weekday_avg = (
    articles_per_day.assign(weekday=articles_per_day.index.day_name())
    .groupby("weekday")["n_articles"].mean()
    .reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])
)

weekday_avg.plot(kind="bar", color="darkorange")
plt.title("Moyenne d'articles par jour de la semaine")
plt.ylabel("Articles (moyenne)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 7. Distribution du `Tone` (tonalité)

Le `Tone` GDELT varie typiquement entre -10 (très négatif) et +10 (très positif), avec une masse centrée autour de 0.
On travaille ici sur un échantillon pour la visualisation (la base complète peut être trop volumineuse pour un histogramme direct).


In [ ]:
tone_sample = con.execute("SELECT Tone FROM gkg USING SAMPLE 500000 ROWS").df()

plt.figure()
sns.histplot(tone_sample["Tone"], bins=100, kde=True, color="slateblue")
plt.title("Distribution du Tone (échantillon de 500k articles)")
plt.xlabel("Tone GDELT")
plt.show()

tone_sample["Tone"].describe()


In [ ]:
tone_monthly = con.execute("""
    SELECT DATE_TRUNC('month', STRPTIME(SUBSTR(DATE, 1, 8), '%Y%m%d')::DATE) AS month,
           AVG(Tone) AS avg_tone,
           COUNT(*) AS n
    FROM gkg
    GROUP BY month
    ORDER BY month
""").df()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(tone_monthly["month"], tone_monthly["avg_tone"], color="firebrick")
ax.axhline(0, color="grey", linestyle="--", linewidth=0.8)
ax.set_title("Tone moyen mensuel — proxy de sentiment médiatique global")
ax.set_ylabel("Tone moyen")
plt.tight_layout()
plt.show()


## 8. Distribution du `WordCount`

In [ ]:
wc_sample = con.execute("SELECT WordCount FROM gkg USING SAMPLE 500000 ROWS").df()
q99 = wc_sample["WordCount"].quantile(0.99)

plt.figure()
sns.histplot(wc_sample.loc[wc_sample["WordCount"] < q99, "WordCount"], bins=80, color="teal")
plt.title("Distribution du WordCount (99e percentile exclu, échantillon)")
plt.xlabel("Nombre de mots")
plt.show()

wc_sample["WordCount"].describe()


## 9. Valeurs manquantes / vides par colonne

In [ ]:
cols_to_check = [
    "GKGRECORDID", "DATE", "SourceCollectionIdentifier", "DocumentIdentifier",
    "EnhancedThemes", "EnhancedLocations", "Persons", "Organizations",
    "TranslationInfo", "Tone", "WordCount", "translated", "SourceCommonName_ID",
]

null_rows = []
for c in cols_to_check:
    n_null = con.execute(
        f"SELECT COUNT(*) FROM gkg WHERE {c} IS NULL OR CAST({c} AS VARCHAR) = ''"
    ).fetchone()[0]
    null_rows.append({"colonne": c, "pct_null_ou_vide": round(100 * n_null / n_rows, 2)})

pd.DataFrame(null_rows).sort_values("pct_null_ou_vide", ascending=False).reset_index(drop=True)


## 10. Thèmes, personnes et organisations les plus cités

Ces champs sont des listes concaténées (`;` entre entités). On travaille sur un **échantillon** pour rester rapide
— pour un traitement exhaustif (utile à la construction d'indicateurs), il faudra industrialiser cet "explode"
en dehors du notebook (Spark, DuckDB `UNNEST`, ou traitement par chunks).


In [ ]:
sample_df = con.execute("""
    SELECT EnhancedThemes, Persons, Organizations
    FROM gkg
    USING SAMPLE 200000 ROWS
""").df()

def top_n_from_semicolon_field(series, n=20, strip_offset=False):
    counter = {}
    for val in series.dropna():
        if not val:
            continue
        for item in val.split(";"):
            item = item.strip()
            if not item:
                continue
            if strip_offset:
                item = item.split(",")[0]
            counter[item] = counter.get(item, 0) + 1
    return pd.Series(counter, dtype="int64").sort_values(ascending=False).head(n)

top_themes = top_n_from_semicolon_field(sample_df["EnhancedThemes"], strip_offset=True)
top_persons = top_n_from_semicolon_field(sample_df["Persons"], strip_offset=False)
top_orgs = top_n_from_semicolon_field(sample_df["Organizations"], strip_offset=False)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

top_themes.sort_values().plot(kind="barh", ax=axes[0], color="seagreen")
axes[0].set_title("Top 20 thèmes GKG")

top_persons.sort_values().plot(kind="barh", ax=axes[1], color="indianred")
axes[1].set_title("Top 20 personnes citées")

top_orgs.sort_values().plot(kind="barh", ax=axes[2], color="goldenrod")
axes[2].set_title("Top 20 organisations citées")

plt.tight_layout()
plt.show()


In [ ]:
# Complexité moyenne d'un article : nombre d'entités détectées
sample_df["n_themes"] = sample_df["EnhancedThemes"].fillna("").apply(lambda x: len([i for i in x.split(";") if i]))
sample_df["n_persons"] = sample_df["Persons"].fillna("").apply(lambda x: len([i for i in x.split(";") if i]))
sample_df["n_orgs"] = sample_df["Organizations"].fillna("").apply(lambda x: len([i for i in x.split(";") if i]))

sample_df[["n_themes", "n_persons", "n_orgs"]].describe()


## 11. Corrélations entre variables numériques

In [ ]:
corr_sample = con.execute("SELECT Tone, WordCount, translated FROM gkg USING SAMPLE 500000 ROWS").df()
corr = corr_sample.corr()

plt.figure(figsize=(5, 4))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Corrélations entre variables numériques")
plt.tight_layout()
plt.show()


In [ ]:
con.close()
print("Connexion DuckDB fermée.")


## 12. Pistes pour construire des indicateurs à partir des GKG events

Quelques directions naturelles à partir de cette base, par ordre croissant de complexité :

- **Indices d'attention médiatique** : volume d'articles par jour filtré sur un thème (`EnhancedThemes` contient `ECON_*`, `CRISISLEX_*`, `ENV_*`...), normalisé par le volume total du jour pour neutraliser les variations de couverture globale.
- **Indices de sentiment thématique** : moyenne du `Tone` pondérée par `WordCount`, calculée séparément pour chaque thème ou pays cité — utile pour suivre l'évolution de la perception médiatique d'un sujet dans le temps.
- **Détection de pics / anomalies** : z-score glissant sur le volume journalier ou sur le `Tone` moyen journalier, pour repérer des ruptures (crises, événements majeurs).
- **Cartographie géographique** : `EnhancedLocations` contient latitude/longitude — on peut agréger par pays/région pour des cartes de chaleur d'attention médiatique.
- **Réseaux de co-occurrence** : construire un graphe (NetworkX) où les nœuds sont des `Persons`/`Organizations` apparaissant dans le même article, pour identifier des clusters d'acteurs liés à un sujet.
- **Décomposition temporelle** : STL ou décomposition saisonnière classique sur les séries de volume/tone pour séparer tendance, saisonnalité et résidu.
- **Biais médiatique cross-source** : comparer le `Tone` moyen par média (`SourceCommonName`) sur un même thème/période pour repérer des différences de traitement éditorial.

Pour l'industrialisation de ces traitements sur l'ensemble de la base (et non un échantillon), DuckDB propose `UNNEST` pour exploser nativement les colonnes `;`-séparées sans repasser par pandas, ce qui sera nettement plus rapide à l'échelle de plusieurs années de données.
